# Plantilla y Generador de Gráficos Estandarizados (Informe Villa Gaucho)

Este cuaderno contiene las herramientas para generar los gráficos de torta, barras horizontales, barras verticales e histogramas continuos con el diseño y formato exacto del **Informe de Villa Gaucho**, con todo el código documentado y comentado paso a paso para que puedas entender y modificar fácilmente cualquier parámetro.

---

### Especificaciones de Diseño Implementadas:
1. **Tipografía y Tamaño de Letra 14**:
   - Todo el gráfico e informe utiliza fuente en tamaño **14** (títulos, ejes, frecuencias absolutas dentro de las partes/barras, porcentajes, etiquetas de categorías y llamadas).
2. **Gráficos de Torta (Formato exacto Villa Gaucho)**:
   - **Sin líneas blancas divisorias** entre las partes de la torta (`edgecolor='none'`).
   - **Frecuencia absoluta adentro** del gráfico (número entero centrado en la cuña en tamaño 14).
   - **Línea de señalización directa (sin subrayar)**:
     - El punto gris sobre la cuña conecta hacia el exterior.
     - La línea guía **arranca exactamente donde comienza el texto de la categoría** (sin extenderse por debajo como subrayado).
     - **Nombre de la categoría arriba** y **porcentaje abajo** (`X,X%`).
   - **Distribución balanceada en el recuadro**: algoritmo de espaciado vertical equitativo para que gráficos con múltiples categorías no queden apretados ni pegados entre sí, manteniéndose visualmente limpios y ordenados.
   - **Regla de más de 5 categorías**:
     - Si la variable tiene más de 5 categorías (o `max_categorias_torta=5`), se recomienda o conmuta automáticamente a un **gráfico de barras horizontales** que utiliza exactamente la misma paleta y orden de colores que el gráfico de torta (1. Azul, 2. Naranja, 3. Verde, 4. Amarillo, 5. Violeta, 6. Beige).
   - **Orden de mayor a menor porcentaje** con la paleta de colores:
     1. **Azul** (`#4285F4`)
     2. **Naranja** (`#FB8C00`)
     3. **Verde** (`#34A853`)
     4. **Amarillo** (`#FBBC05`)
     5. **Violeta** (`#8E24AA`)
     6. **Beige** (`#D7CCC8`)
   - Para preguntas de **positivo y negativo** (ej: Sí / No): **Positivo = Azul (`#4285F4`)**, **Negativo = Naranja (`#FB8C00`)**.
3. **Gráficos de Barras (Horizontales y Verticales)**:
   - **Regla del 2%**:
     - Si el porcentaje es **menor a 2%** (`porcentaje < 2.0`), la frecuencia absoluta y el porcentaje aparecen **afuera de la barra** (`"X (Y,Y%)"`), evitando saturación visual o números que no caben.
     - Si el porcentaje es **mayor o igual a 2%**, la frecuencia absoluta se muestra **adentro de la barra** (en blanco y negrita, tamaño 14) y el porcentaje afuera.
   - **Color de barras de un solo tono**: **`#1b7882`**.
   - Opción `usar_colores_torta=True` para usar exactamente los mismos colores del gráfico de torta en caso de requerir barras multicolores categóricas (como cuando hay más de 5 categorías).
4. **Histogramas Continuos (Variables Cuantitativas Continuas, ej. Edad, Ingresos, Tiempo)**:
   - **Barras continuas unidas** (`width=1.0` o `rwidth=1.0`), **sin separación física entre barras**.
   - La división entre barras no es un espacio en blanco, sino el **borde de la barra con el mismo color pero un 40% más oscuro** (`#10484e`), con un ancho aproximado de **`0.1`**.
   - Parámetros documentados: `ancho_borde=0.1` y `factor_oscuridad=0.40`.


In [ ]:
# ==============================================================================
# CONFIGURACIÓN GLOBAL Y LIBRERÍAS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Configuramos matplotlib para que TODOS los textos por defecto tengan tamaño 14
plt.rcParams['font.size'] = 14          # Tamaño de letra base
plt.rcParams['axes.titlesize'] = 14     # Tamaño del título del gráfico
plt.rcParams['axes.labelsize'] = 14     # Tamaño de etiquetas en los ejes X e Y
plt.rcParams['xtick.labelsize'] = 14    # Tamaño de los valores del eje X
plt.rcParams['ytick.labelsize'] = 14    # Tamaño de los valores del eje Y
plt.rcParams['legend.fontsize'] = 14    # Tamaño de la leyenda
plt.rcParams['figure.titlesize'] = 14   # Tamaño del título de la figura

# ==============================================================================
# PALETA DE COLORES ESTANDARIZADA
# ==============================================================================
# Paleta para tortas: asignada estrictamente de mayor a menor porcentaje:
# 1. Azul, 2. Naranja, 3. Verde, 4. Amarillo, 5. Violeta, 6. Beige
PALETA_TORTA = [
    '#4285F4',  # 1. Azul (Color institucional / Google)
    '#FB8C00',  # 2. Naranja
    '#34A853',  # 3. Verde
    '#FBBC05',  # 4. Amarillo
    '#8E24AA',  # 5. Violeta
    '#D7CCC8'   # 6. Beige
]

# Color específico para gráficos de barra e histogramas continuos
COLOR_BARRAS_UNICO = '#1b7882'

# Colores para preguntas binarias (Positivo vs Negativo)
COLOR_POSITIVO = '#4285F4'  # Azul para respuesta afirmativa / positiva
COLOR_NEGATIVO = '#FB8C00'  # Naranja para respuesta negativa

# ==============================================================================
# FUNCIÓN AUXILIAR: OSCURECER COLOR
# ==============================================================================
def oscurecer_color(color_hex, factor_oscuridad=0.40):
    """
    Toma un color hexadecimal y devuelve el mismo color oscurecido en la proporción indicada.
    Por ejemplo, con factor_oscuridad=0.40, devuelve el color 40% más oscuro.
    """
    rgb = mcolors.to_rgb(color_hex)
    return tuple(c * (1.0 - factor_oscuridad) for c in rgb)


## 1. Función para Gráficos de Torta Estandarizados (con Conmutación Automática si > 5 categorías)
- Si tiene más de 5 categorías (`len(categorias) > max_categorias_torta` y `conmutar_si_muchas_categorias=True`), genera un gráfico de **barras horizontales con los mismos colores de la torta**.
- Si tiene 5 o menos categorías, genera el gráfico de torta estandarizado.
- La línea guía **conecta directamente al punto donde comienza el texto** (sin subrayar).


In [ ]:
def graficar_torta(categorias, valores, titulo="", es_binario=False, orden_positivo_primero=True, figsize=(11, 7), distancia_etiquetas_y=0.45, max_categorias_torta=5, conmutar_si_muchas_categorias=True):
    """
    Genera un gráfico de torta con el formato exacto del Informe de Villa Gaucho.
    
    Parámetros:
    -----------
    categorias : list o pd.Series
        Nombres de las categorías a graficar.
    valores : list o pd.Series
        Frecuencias absolutas de cada categoría.
    titulo : str, opcional
        Título principal del gráfico.
    es_binario : bool, por defecto False
        Indica si es una pregunta binaria (Positivo = Azul, Negativo = Naranja).
    orden_positivo_primero : bool, por defecto True
        Si es binario, ubica primero la respuesta afirmativa.
    figsize : tuple, por defecto (11, 7)
        Dimensiones (ancho, alto) de la figura.
    distancia_etiquetas_y : float, por defecto 0.45
        Separación vertical entre etiquetas para mantener el gráfico visualmente espaciado.
    max_categorias_torta : int, por defecto 5
        Cantidad máxima de categorías recomendada para torta antes de pasar a barras horizontales.
    conmutar_si_muchas_categorias : bool, por defecto True
        Si es True y hay más de 5 categorías, genera un gráfico de barras horizontales con los colores de la torta.
    """
    df_temp = pd.DataFrame({'categoria': [str(c) for c in categorias], 'fi': list(valores)})

    # Si hay más de 5 categorías y la opción está habilitada, conmutar a barras horizontales con paleta de torta
    if conmutar_si_muchas_categorias and len(df_temp) > max_categorias_torta and not es_binario:
        print(f"[INFO] La variable tiene {len(df_temp)} categorías (> {max_categorias_torta}). Se grafica con barras horizontales usando la paleta de torta.")
        graficar_barras_horizontales(
            etiquetas=df_temp['categoria'],
            valores=df_temp['fi'],
            titulo=titulo,
            usar_colores_torta=True,
            ordenar_mayor_a_menor=True,
            figsize=(13, max(6, len(df_temp) * 0.8))
        )
        return

    # Asignación de colores y orden
    if es_binario:
        def es_pos(val):
            val_clean = str(val).strip().lower()
            return val_clean in ['si', 'sí', 'positivo', 'true', 'afirma', 'posee', 'tiene', 'concurre']

        df_temp['es_pos'] = df_temp['categoria'].apply(es_pos)
        if orden_positivo_primero:
            df_temp = df_temp.sort_values(by='es_pos', ascending=False).reset_index(drop=True)
        colores = [COLOR_POSITIVO if p else COLOR_NEGATIVO for p in df_temp['es_pos']]
    else:
        df_temp = df_temp.sort_values(by='fi', ascending=False).reset_index(drop=True)
        colores = [PALETA_TORTA[i % len(PALETA_TORTA)] for i in range(len(df_temp))]

    cats = df_temp['categoria'].tolist()
    vals = df_temp['fi'].values
    total = sum(vals)
    pcts = (vals / total) * 100 if total > 0 else np.zeros_like(vals)

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(aspect="equal"))

    # Torta sin líneas divisorias blancas
    wedges, texts, autotexts = ax.pie(
        vals,
        autopct='%1.0f',
        pctdistance=0.68,
        colors=colores,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=1, edgecolor='none')
    )

    # Frecuencia absoluta adentro de cada porción (tamaño 14)
    for at, val in zip(autotexts, vals):
        at.set_text(str(int(val)))
        at.set_color('#202124')
        at.set_fontsize(14)
        at.set_fontweight('medium')

    wedge_items = []
    for i, p in enumerate(wedges):
        ang = (p.theta2 - p.theta1) / 2.0 + p.theta1
        rad = np.deg2rad(ang)
        x = np.cos(rad)
        y = np.sin(rad)
        lado = 1 if x >= 0 else -1
        wedge_items.append({
            'index': i,
            'ang': ang,
            'rad': rad,
            'x': x,
            'y': y,
            'lado': lado,
            'y_afuera': 1.15 * y
        })

    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-1.6, 1.6)

    # Distribución vertical balanceada
    for lado in [1, -1]:
        group = [item for item in wedge_items if item['lado'] == lado]
        if not group:
            continue
        group.sort(key=lambda item: item['y'])
        k = len(group)
        if k == 1:
            group[0]['y_afuera'] = group[0]['y'] * 1.15
        else:
            span = (k - 1) * distancia_etiquetas_y
            avg_y = np.mean([item['y'] for item in group])
            start_y = avg_y - (span / 2.0)
            start_y = max(-1.30, min(1.30 - span, start_y))
            for idx_g, item in enumerate(group):
                item['y_afuera'] = start_y + idx_g * distancia_etiquetas_y

    for item in wedge_items:
        i = item['index']
        rad = item['rad']
        x_punto = 0.90 * np.cos(rad)
        y_punto = 0.90 * np.sin(rad)

        lado = item['lado']
        y_afuera = item['y_afuera']
        ha = 'left' if lado == 1 else 'right'
        pct_str = f"{pcts[i]:.1f}%".replace('.', ',')

        x_texto = 1.95 * lado

        # 1. Nombre de la categoría ARRIBA de la línea
        ax.text(
            x_texto, y_afuera + 0.03,
            cats[i],
            horizontalalignment=ha,
            verticalalignment='bottom',
            fontsize=14,
            color='#202124',
            fontweight='medium'
        )

        # 2. Porcentaje ABAJO de la línea
        ax.text(
            x_texto, y_afuera - 0.03,
            pct_str,
            horizontalalignment=ha,
            verticalalignment='top',
            fontsize=14,
            color='#757575',
            fontweight='normal'
        )

        # 3. Línea que conecta la porción con donde arranca el texto (SIN SUBRAYAR)
        ax.plot([x_punto, x_texto], [y_punto, y_afuera], color='#9e9e9e', lw=1.2)
        # Punto gris indicador
        ax.plot(x_punto, y_punto, marker='o', markersize=4, color='#9e9e9e')

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=25, fontweight='bold', color='#202124')

    plt.subplots_adjust(left=0.08, right=0.92, top=0.88, bottom=0.10)
    plt.show()


## 2. Gráficos de Barras Estandarizados (Horizontales y Verticales con Regla del 2%)
- **Regla del 2%**: Si el porcentaje es menor a 2%, la frecuencia absoluta se coloca **afuera de la barra** (`"X (Y,Y%)"`). Si es >= 2%, la frecuencia absoluta se coloca **adentro de la barra** en blanco y negrita.
- Soporta `usar_colores_torta=True` para barras multicolores ordenadas con la paleta de la torta.


In [ ]:
def graficar_barras_horizontales(etiquetas, valores, titulo="", xlabel="Cantidad de respuestas", ylabel="", sufijo_etiqueta="", usar_colores_torta=False, ordenar_mayor_a_menor=False, figsize=(12, 6)):
    """
    Genera un gráfico de barras horizontales estandarizado:
    - Si porcentaje < 2%, frecuencia y porcentaje van afuera de la barra.
    - Si porcentaje >= 2%, frecuencia adentro de la barra (blanco, negrita) y porcentaje afuera.
    - Si usar_colores_torta=True, asigna los colores de la torta de mayor a menor porcentaje.
    """
    df_temp = pd.DataFrame({'etiqueta': [str(e) for e in etiquetas], 'fi': list(valores)})
    
    if ordenar_mayor_a_menor:
        df_temp = df_temp.sort_values(by='fi', ascending=False).reset_index(drop=True)
        # En barh, invertimos para que la mayor categoría quede arriba
        df_temp = df_temp.iloc[::-1].reset_index(drop=True)

    valores = np.array(df_temp['fi'].values, dtype=float)
    etiquetas = df_temp['etiqueta'].tolist()
    total = np.sum(valores)
    porcentajes = (valores / total) * 100 if total > 0 else np.zeros_like(valores)
    n = len(etiquetas)

    fig, ax = plt.subplots(figsize=figsize)

    if usar_colores_torta:
        if ordenar_mayor_a_menor:
            ranks = n - 1 - np.arange(n)
            colores = [PALETA_TORTA[r % len(PALETA_TORTA)] for r in ranks]
        else:
            colores = [PALETA_TORTA[i % len(PALETA_TORTA)] for i in range(n)]
    else:
        colores = COLOR_BARRAS_UNICO

    barras = ax.barh(
        range(n),
        valores,
        color=colores,
        height=0.68,
        edgecolor='none'
    )

    ax.set_yticks(range(n))
    if sufijo_etiqueta:
        labels_y = [f"{lbl} {sufijo_etiqueta}" if int(lbl) != 1 else f"{lbl} {sufijo_etiqueta.rstrip('s')}" for lbl in etiquetas]
    else:
        labels_y = [str(lbl) for lbl in etiquetas]
    ax.set_yticklabels(labels_y, fontsize=14)

    max_val = max(valores) if len(valores) > 0 else 1
    offset_texto = max_val * 0.025

    for barra, cantidad, porcentaje in zip(barras, valores, porcentajes):
        x = barra.get_width()
        y = barra.get_y() + barra.get_height() / 2.0

        # REGLA DEL 2%:
        if porcentaje < 2.0:
            # Si es menor al 2%, la frecuencia absoluta aparece AFUERA de la barra junto al porcentaje
            ax.text(
                x + offset_texto, y,
                f"{int(cantidad)} ({porcentaje:.1f}%)".replace('.', ','),
                ha='left', va='center',
                color='#202124', fontsize=14, fontweight='medium'
            )
        else:
            # Si es mayor o igual al 2%, frecuencia adentro y porcentaje afuera
            ax.text(
                x / 2.0, y, str(int(cantidad)),
                ha='center', va='center',
                color='white', fontweight='bold', fontsize=14
            )
            ax.text(
                x + offset_texto, y,
                f"{porcentaje:.1f}%".replace('.', ','),
                ha='left', va='center',
                color='#202124', fontsize=14
            )

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=15, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=14, labelpad=10)

    ax.set_xlim(0, max_val * 1.25)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()


def graficar_barras_verticales(etiquetas, valores, titulo="", xlabel="Cantidad de personas", ylabel="Cantidad de respuestas", figsize=(9, 6)):
    """
    Genera un gráfico de barras VERTICAL estandarizado:
    - Si porcentaje < 2%, frecuencia y porcentaje van afuera de la barra (arriba).
    - Si porcentaje >= 2%, frecuencia adentro de la barra (blanco, negrita) y porcentaje arriba.
    - Color #1b7882.
    """
    valores = np.array(valores, dtype=float)
    total = np.sum(valores)
    porcentajes = (valores / total) * 100 if total > 0 else np.zeros_like(valores)
    n = len(etiquetas)
    x_pos = np.arange(n)

    fig, ax = plt.subplots(figsize=figsize)

    barras = ax.bar(
        x_pos,
        valores,
        color=COLOR_BARRAS_UNICO,
        width=0.62,
        edgecolor='none'
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(lbl) for lbl in etiquetas], fontsize=14)

    max_h = max(valores) if len(valores) > 0 else 1

    for barra, cantidad, porcentaje in zip(barras, valores, porcentajes):
        h = barra.get_height()
        x = barra.get_x() + barra.get_width() / 2.0

        # REGLA DEL 2%:
        if porcentaje < 2.0:
            # Afuera de la barra (arriba) con frecuencia y porcentaje
            ax.text(
                x, h + (max_h * 0.02),
                f"{int(cantidad)}\n({porcentaje:.1f}%)".replace('.', ','),
                ha='center', va='bottom',
                color='#202124', fontsize=14, fontweight='medium'
            )
        else:
            # Frecuencia adentro y porcentaje arriba
            ax.text(
                x, h / 2.0, str(int(cantidad)),
                ha='center', va='center',
                color='white', fontweight='bold', fontsize=14
            )
            ax.text(
                x, h + (max_h * 0.02),
                f"{porcentaje:.1f}%".replace('.', ','),
                ha='center', va='bottom',
                color='#202124', fontsize=14
            )

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=18, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=14, labelpad=10)

    ax.set_ylim(0, max_h * 1.25)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()


## 3. Histograma Continuo para Variables Cuantitativas Continuas
- Barras continuas unidas sin separación (`width=1.0`).
- Borde divisorio con el mismo color pero 40% más oscuro (`factor_oscuridad=0.40`) y ancho `0.1`.


In [ ]:
def graficar_histograma_intervalos(intervalos, frecuencias, titulo="", xlabel="Rango de edad", ylabel="Cantidad de menores", ancho_borde=0.1, factor_oscuridad=0.40, figsize=(10, 6)):
    """
    Genera un histograma continuo (barras unidas sin separación) a partir de rangos/intervalos y frecuencias.
    """
    frecuencias = np.array(frecuencias, dtype=float)
    total = sum(frecuencias)
    porcentajes = (frecuencias / total) * 100 if total > 0 else np.zeros_like(frecuencias)
    x_pos = np.arange(len(intervalos))

    fig, ax = plt.subplots(figsize=figsize)
    color_borde_separacion = oscurecer_color(COLOR_BARRAS_UNICO, factor_oscuridad=factor_oscuridad)

    barras = ax.bar(
        x_pos,
        frecuencias,
        width=1.0,                              # Continuo sin huecos
        color=COLOR_BARRAS_UNICO,                # #1b7882
        edgecolor=color_borde_separacion,       # Borde 40% más oscuro
        linewidth=ancho_borde,                  # Ancho de separación (~0.1)
        align='center'
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(intervalos, fontsize=14, rotation=0)

    max_h = max(frecuencias) if len(frecuencias) > 0 else 1

    for barra, cant, pct in zip(barras, frecuencias, porcentajes):
        h = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width() / 2.0,
            h + (max_h * 0.02),
            f"{int(cant)}\n({pct:.1f}%)".replace('.', ','),
            ha='center', va='bottom',
            fontsize=14, color='#202124', fontweight='medium'
        )

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=18, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=14, labelpad=10)

    ax.set_ylim(0, max_h * 1.25)
    ax.set_xlim(-0.5, len(intervalos) - 0.5)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()


## 4. Ejemplos de Uso
A continuación se muestran ejemplos prácticos de cada tipo de gráfico con las nuevas funcionalidades:
1. Torta con línea directa sin subrayar.
2. Torta que conmuta automáticamente a barras horizontales por tener más de 5 categorías.
3. Barras horizontales y verticales con la regla de porcentajes menores a 2%.
4. Histograma continuo.


In [ ]:
# Ejemplo 1: Torta normal (4 categorías) con línea directa que no subraya
categorias_ejemplo = ['Propio', 'Alquilado', 'Prestado', 'No responden']
valores_ejemplo = [158, 17, 4, 3]

graficar_torta(
    categorias=categorias_ejemplo,
    valores=valores_ejemplo,
    titulo="¿El lugar de residencia es propio, alquilado o prestado?"
)


In [ ]:
# Ejemplo 2: Variable con más de 5 categorías -> Conmuta automáticamente a barras con colores de torta
categorias_muchas = [
    'Cocina a gas',
    'Cocina a gas, Microondas',
    'Cocina a gas, Microondas, Horno eléctrico',
    'Horno eléctrico',
    'Microondas',
    'Cocina a gas, Horno eléctrico',
    'Cocina a leña, Microondas, Horno eléctrico'
]
valores_muchas = [36, 10, 2, 1, 1, 1, 1]

graficar_torta(
    categorias=categorias_muchas,
    valores=valores_muchas,
    titulo="Equipamiento que tienen en el hogar para cocinar (Más de 5 categorías)"
)


In [ ]:
# Ejemplo 3: Barras horizontales con valores donde el porcentaje es < 2% (van afuera de la barra)
dias = [0, 1, 2, 3, 4, 5, 6, 7]
respuestas_dias = [10, 1, 5, 5, 5, 0, 4, 130]

graficar_barras_horizontales(
    etiquetas=dias,
    valores=respuestas_dias,
    titulo="Distribución de días en los que se desayuna en el hogar",
    xlabel="Cantidad de respuestas",
    ylabel="Cantidad de días",
    sufijo_etiqueta="días"
)


In [ ]:
# Ejemplo 4: Barras verticales con valores donde el porcentaje es < 2%
personas = ['1', '2', '3', '4']
respuestas_personas = [33, 13, 4, 1] # 1 caso es < 2%

graficar_barras_verticales(
    etiquetas=personas,
    valores=respuestas_personas,
    titulo="Cantidad de personas que definen qué se cocina",
    xlabel="Cantidad de personas",
    ylabel="Cantidad de respuestas"
)


In [ ]:
# Ejemplo 5: Histograma continuo de edades
rangos_edad = ['0 a 2 años', '3 a 5 años', '6 a 11 años', '12 a 18 años']
cantidades_menores = [13, 14, 56, 58]

graficar_histograma_intervalos(
    intervalos=rangos_edad,
    frecuencias=cantidades_menores,
    titulo="Edad de los menores en el hogar (agrupada en rangos)",
    xlabel="Rango de edad",
    ylabel="Cantidad de menores"
)


## 5. Integración directa con `datos.csv`


In [ ]:
# Prueba de carga directa con datos.csv
try:
    df_datos = pd.read_csv('datos.csv')
    print("Dataset cargado correctamente. Filas:", len(df_datos), "| Columnas:", len(df_datos.columns))

    col_sal = '1.7 Habitualmente cuando se cocina en el hogar, ¿se le agrega sal a las comidas durante la cocción?'
    if col_sal in df_datos.columns:
        conteo_sal = df_datos[col_sal].value_counts()
        graficar_torta(
            categorias=conteo_sal.index,
            valores=conteo_sal.values,
            titulo="Agregado de sal durante la cocción",
            es_binario=True
        )
except FileNotFoundError:
    print("Archivo datos.csv no encontrado en el directorio actual.")
